# TOTNet Ball-Tracking Inference — Google Colab

**Purpose:** Run TOTNet inference on a custom video, save annotated frames and an MP4 to Google Drive.

**Inputs required:**
- A video file (`.mp4` or `.mov`), preferably ~1280×720 @ ~38 fps
- Google Drive mounted at `/content/drive` for output saving

**Requirements:** GPU runtime — *Runtime → Change runtime type → T4 GPU*

---
**Quick-start — run cells in order:**
1. Cell 1 — GPU check (must pass before continuing)
2. Cell 2 — Mount Google Drive
3. Cell 3 — Install dependencies
4. Cell 4 — Clone TOTNet repo
5. **Cell 5 — Edit your paths**, then run all remaining cells


In [ ]:
# Cell 1 — Environment check
import torch, sys, platform

cuda_ok = torch.cuda.is_available()
print('CUDA available :', cuda_ok)
if cuda_ok:
    print('GPU            :', torch.cuda.get_device_name(0))
    print('CUDA version   :', torch.version.cuda)
else:
    raise RuntimeError(
        'No GPU detected. '
        'Go to Runtime → Change runtime type → Hardware accelerator: GPU (T4).'
    )
print('Python         :', sys.version)
print('PyTorch        :', torch.__version__)


In [ ]:
# Cell 2 — Mount Google Drive (outputs will be saved here)
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')


In [ ]:
# Cell 3 — Install dependencies
# PyTorch 2.4.1 + CUDA 12.1 matches current Colab T4/A100 runtimes.
# einops is required by the TOTNet model architecture.
%pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 --extra-index-url https://download.pytorch.org/whl/cu121
%pip install -q easydict matplotlib opencv-python scikit-learn cython pycocotools tqdm scipy ninja tensorboard ptflops einops
!apt-get -y -q install ffmpeg
print('All dependencies installed.')


In [ ]:
# Cell 4 — Clone TOTNet repository (contains pretrained weights)
#
# We clone deepdewdeep/TOTNet, which is the repository containing the
# bug-fixed demo.py and the pretrained weights in the weights/ folder.
import os, subprocess

REPO_DIR = '/content/TOTNet'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/deepdewdeep/TOTNet.git', REPO_DIR], check=True)
else:
    print(f'Repo already cloned at {REPO_DIR} — pulling latest changes')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

print('\nPretrained weight folders:')
print(os.listdir(os.path.join(REPO_DIR, 'weights')))


## Cell 5 — User Configuration ← **edit this cell**

Set:
- `VIDEO_PATH` to your input video (upload via the Files panel, or reference a Drive path)
- `SPORT` to match the type of ball in your video
- `OUTPUT_DRIVE_DIR` to where results should land on your Drive


In [ ]:
# Cell 5 — User configuration  ← EDIT THIS CELL
import os, glob

# ── Input video ──────────────────────────────────────────────────────────────
# Upload your video via Colab's Files panel (left sidebar) and set the path,
# OR reference a file already on Google Drive, e.g.:
#   VIDEO_PATH = '/content/drive/MyDrive/my_match.mp4'
VIDEO_PATH = '/content/input.mp4'   # <-- set this

# ── Sport / weight selection ─────────────────────────────────────────────────
# 'tennis'    → TOTNet_Tennis weights   (lawn tennis ball)
# 'badminton' → TOTNet_Badminton weights
# 'tta'       → TOTNet_TTA weights      (table tennis, TTA dataset)
SPORT = 'tennis'   # <-- set this

# ── Output destination on Google Drive ───────────────────────────────────────
OUTPUT_DRIVE_DIR = '/content/drive/MyDrive/TOTNet_outputs'   # <-- set this

# ── Inference label (used as the output sub-folder name) ─────────────────────
DEMO_NAME = 'my_demo'   # alphanumeric + underscores only

# ─────────────────────────────────────────────────────────────────────────────
#  Fixed model settings — must match the pretrained weights
# ─────────────────────────────────────────────────────────────────────────────
NUM_FRAMES = 5      # sliding-window size (matches the (5) in the weight folder names)
IMG_H, IMG_W = 288, 512   # model input resolution (height, width)

# ─────────────────────────────────────────────────────────────────────────────
#  Resolve weight path automatically from the cloned repo
# ─────────────────────────────────────────────────────────────────────────────
REPO_DIR = '/content/TOTNet'
_sport_prefix = {
    'tennis':    'TOTNet_Tennis_',
    'badminton': 'TOTNet_Badminton_',
    'tta':       'TOTNet_TTA_',
}
assert SPORT in _sport_prefix, (
    f'Unknown SPORT={SPORT!r}. Choose from: {list(_sport_prefix)}'
)
_candidates = glob.glob(
    os.path.join(REPO_DIR, 'weights', f'{_sport_prefix[SPORT]}*', '*_best.pth')
)
assert _candidates, (
    f'No *_best.pth weight found for SPORT={SPORT!r} under {REPO_DIR}/weights/. '
    f'Check the repo cloned successfully in Cell 4.'
)
WEIGHT_PATH = sorted(_candidates)[0]

print(f'Video      : {VIDEO_PATH}')
print(f'Sport      : {SPORT}')
print(f'Weights    : {WEIGHT_PATH}')
print(f'Drive dir  : {OUTPUT_DRIVE_DIR}')
print(f'Demo name  : {DEMO_NAME}')


In [ ]:
# Cell 6 — Pre-flight checks (detects problems before wasting inference time)
import os, cv2

# 1. Video exists
assert os.path.isfile(VIDEO_PATH), (
    f'Video not found: {VIDEO_PATH!r}\n'
    'Upload your video via the Colab Files panel or set VIDEO_PATH to a Drive path.'
)
print(f'[OK] Video found  : {VIDEO_PATH}  ({os.path.getsize(VIDEO_PATH) // 1024:,} KB)')

# 2. Extension
ext = os.path.splitext(VIDEO_PATH)[1].lower()
assert ext in ('.mp4', '.mov'), f'Unsupported extension: {ext!r}. Use .mp4 or .mov.'
print(f'[OK] Extension    : {ext}')

# 3. Weights exist
assert os.path.isfile(WEIGHT_PATH), f'Weight file not found: {WEIGHT_PATH!r}'
print(f'[OK] Weights found: {WEIGHT_PATH}')

# 4. Video properties
cap = cv2.VideoCapture(VIDEO_PATH)
try:
    fps = cap.get(cv2.CAP_PROP_FPS)
    nf  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    vw  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    vh  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
finally:
    cap.release()
print(f'[OK] Video props  : {vw}x{vh} @ {fps:.1f} fps, {nf} frames')
assert nf >= NUM_FRAMES, (
    f'Video has only {nf} frames but NUM_FRAMES={NUM_FRAMES}. Use a longer clip.'
)

# 5. Google Drive output directory
os.makedirs(OUTPUT_DRIVE_DIR, exist_ok=True)
print(f'[OK] Drive dir    : {OUTPUT_DRIVE_DIR}')

print('\nAll pre-flight checks passed. Ready to run inference.')


In [ ]:
# Cell 7 — Run inference
#
# Output lands in: /content/results/demo/<DEMO_NAME>/
#   frame/           annotated JPEG frames
#   result.mp4       assembled annotated video
#
# The predicted ball position (magenta dot) is drawn at the ORIGINAL
# frame resolution after scaling coordinates from model space (288x512).

import subprocess, sys, os

REPO_DIR = '/content/TOTNet'
WORK_DIR = '/content'   # results will appear at /content/results/demo/<DEMO_NAME>/

cmd = [
    sys.executable,
    os.path.join(REPO_DIR, 'src', 'demo.py'),
    '--model_choice',    'motion_light',
    '--dataset_choice',  'tt',        # 'tt' selects Video_Loader for .mp4/.mov input
    '--video_path',      VIDEO_PATH,
    '--pretrained_path', WEIGHT_PATH,
    '--num_frames',      str(NUM_FRAMES),
    '--img_size',        str(IMG_H), str(IMG_W),
    '--gpu_idx',         '0',
    '--save_demo_output',
    '--output_format',   'video',
    '--working-dir',     WORK_DIR,
    '--saved_fn',        DEMO_NAME,
]

print('Command:', ' '.join(cmd))
print('=' * 70)

proc = subprocess.run(
    cmd,
    cwd=os.path.join(REPO_DIR, 'src'),  # demo.py uses relative imports; must run from src/
    text=True, capture_output=True
)

# Print stdout/stderr fully to expose any silent failures
stdout_tail = proc.stdout[-8000:] if len(proc.stdout) > 8000 else proc.stdout
print(stdout_tail)
if proc.stderr.strip():
    print('--- STDERR ---')
    print(proc.stderr[-4000:] if len(proc.stderr) > 4000 else proc.stderr)

if proc.returncode != 0:
    raise RuntimeError(
        f'demo.py exited with code {proc.returncode}. '
        'See the output above for details.'
    )

print('\n[OK] Inference completed.')


In [ ]:
# Cell 8 — Verify output (explicit anti-silent-failure checks)
#
# The draft notebook had a known failure mode where inference ran but
# produced unannotated frames or an empty video.  These checks catch that.

import os, glob, cv2
import matplotlib.pyplot as plt

FRAMES_DIR   = os.path.join('/content', 'results', 'demo', DEMO_NAME, 'frame')
OUTPUT_VIDEO = os.path.join('/content', 'results', 'demo', DEMO_NAME, 'result.mp4')

# ── Check 1: annotated frames were saved ────────────────────────────────────
frame_files = sorted(glob.glob(os.path.join(FRAMES_DIR, '*.jpg')))
assert len(frame_files) > 0, (
    f'No annotated frames found in {FRAMES_DIR}!\n'
    'Inference may have been silently skipped — re-read the Cell 7 output carefully.'
)
print(f'[OK] Annotated frames : {len(frame_files)}')

# ── Check 2: output video was assembled ─────────────────────────────────────
assert os.path.isfile(OUTPUT_VIDEO), (
    f'Output video not found: {OUTPUT_VIDEO}\n'
    'ffmpeg may have failed — check STDERR in Cell 7.'
)
video_kb = os.path.getsize(OUTPUT_VIDEO) // 1024
assert video_kb > 10, (
    f'Output video is suspiciously small ({video_kb} KB) — it may be corrupt.\n'
    'Try re-running Cell 7.'
)
print(f'[OK] Output video     : {OUTPUT_VIDEO}  ({video_kb:,} KB)')

# ── Check 3: spot-check first & last annotated frames ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, fp, label in zip(
    axes,
    [frame_files[0], frame_files[-1]],
    ['First annotated frame', 'Last annotated frame'],
):
    img = cv2.cvtColor(cv2.imread(fp), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f'{label}\n{os.path.basename(fp)}')
    ax.axis('off')
plt.suptitle(
    'Predicted ball positions (magenta dot).\n'
    'If no dot is visible, the ball may be absent or occluded in those frames.',
    fontsize=11,
)
plt.tight_layout()
plt.show()

print('\n[OK] Output verification passed.')


In [ ]:
# Cell 9 — Preview annotated video inline
from IPython.display import Video, display
display(Video(OUTPUT_VIDEO, embed=True, width=720))


In [ ]:
# Cell 10 — Save outputs to Google Drive
import os, shutil, glob

FRAMES_DIR   = os.path.join('/content', 'results', 'demo', DEMO_NAME, 'frame')
OUTPUT_VIDEO = os.path.join('/content', 'results', 'demo', DEMO_NAME, 'result.mp4')

drive_demo_dir = os.path.join(OUTPUT_DRIVE_DIR, DEMO_NAME)
os.makedirs(drive_demo_dir, exist_ok=True)

# Copy annotated video
dst_video = os.path.join(drive_demo_dir, 'result.mp4')
shutil.copy2(OUTPUT_VIDEO, dst_video)
print(f'[Drive] Video  → {dst_video}')

# Copy annotated frames folder
dst_frames = os.path.join(drive_demo_dir, 'frames')
if os.path.isdir(dst_frames):
    shutil.rmtree(dst_frames)
shutil.copytree(FRAMES_DIR, dst_frames)
n_saved = len(glob.glob(os.path.join(dst_frames, '*.jpg')))
print(f'[Drive] Frames → {dst_frames}  ({n_saved} files)')

print('\n[OK] All outputs saved to Google Drive.')
